# Notebook 03: `Voxel51/WLASL` Top-100 Visual Baselines (Validation Benchmark)

**Runtime:** Works fast on **Colab CPU (`~3–6 min` total)** or **T4 GPU (`< 1 min`)** using cached `(N, 32, 2048)` features.  
**Strict Split Discipline:** Trains ONLY on `train` (`672` clips, `31` signers) and benchmarks strictly on `val` (`218` clips, `17` unseen signers). **`test` (`230` clips) remains locked until Notebook 05.**

### Experiments in This Notebook
0. **Sanity Floors:** Uniform Random Chance (`1.0%` Top-1) & Majority Class Baseline + Mean-Pooled Linear Classifier (`Linear(2048 -> 100)`).
1. **`visual_lstm_last`:** Classic `ResNet50 + LSTM` visual baseline (`hidden=256`, `layers=2`, `pooling="last"`, no augmentation).
2. **`visual_bilstm_attn`:** Bidirectional LSTM (`hidden=256`, `layers=2`, `pooling="attention"`) + temporal augmentations + label smoothing (`0.05`).
3. **`visual_transformer_attn`:** Lightweight Temporal Transformer (`embed_dim=256`, `layers=2`, `heads=4`, `pooling="attention"`) + augmentations + label smoothing (`0.05`).

All checkpoints (`best_model.pt`) match [`src/inference/predictor.py`](../src/inference/predictor.py) `strict=True` and log to Drive `experiments/results_summary.csv`.


## 1. Environment Setup & Google Drive Mount


In [ ]:
import contextlib
import json
import math
import random
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import drive as colab_drive
    IN_COLAB = True
except ImportError:
    colab_drive = None
    IN_COLAB = False

if IN_COLAB and colab_drive is not None:
    if Path("/content/drive/MyDrive").exists():
        print("Google Drive already mounted at /content/drive.")
    else:
        colab_drive.mount("/content/drive")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {DEVICE} | IN_COLAB={IN_COLAB}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

## 2. Load Cached Features (`train` / `val` Only; `test` Stays Locked)


In [ ]:
# ==============================================================================
# CONFIGURATION & DATA LOADING
# ==============================================================================
SEED = 42
NUM_CLASSES = 100
FEATURE_DIM = 2048
BATCH_SIZE = 32
EPOCHS = 70
PATIENCE = 20

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

if IN_COLAB and Path("/content/drive/MyDrive").exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100")
else:
    DRIVE_ROOT = Path.cwd() / "data" / "voxel51_wlasl100"

META_DIR = DRIVE_ROOT / "metadata"
FEAT_DIR = DRIVE_ROOT / "features"
CKPT_DIR = DRIVE_ROOT / "checkpoints"
EXP_DIR = DRIVE_ROOT / "experiments"
PLOTS_DIR = DRIVE_ROOT / "reports" / "plots"

for d in (CKPT_DIR, EXP_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

class_to_id = {str(k): int(v) for k, v in json.loads((META_DIR / "class_to_id.json").read_text(encoding="utf-8")).items()}
id_to_class = {int(k): str(v) for k, v in json.loads((META_DIR / "id_to_class.json").read_text(encoding="utf-8")).items()}
assert len(class_to_id) == NUM_CLASSES

def load_split(sp: str):
    x = np.asarray(np.load(FEAT_DIR / f"X_{sp}.npy"), dtype=np.float32)
    y = np.asarray(np.load(FEAT_DIR / f"y_{sp}.npy"), dtype=np.int64)
    done = np.load(FEAT_DIR / f"done_{sp}.npy")
    assert bool(done.all()), f"{sp} extraction incomplete!"
    assert np.isfinite(x).all(), f"{sp} contains non-finite values!"
    return x, y

X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
print(f"Loaded Train: {X_train.shape} ({len(np.unique(y_train))} classes)")
print(f"Loaded Val  : {X_val.shape} ({len(np.unique(y_val))} classes) | Test split kept locked.")

## 3. Dataset, Augmentations & Metrics


In [ ]:
# ==============================================================================
# DATASET, AUGMENTATION & METRICS (ALIGNED WITH src/)
# ==============================================================================
def augment_feature_sequence(
    features: np.ndarray,
    *,
    crop_ratio_range: tuple[float, float] = (0.85, 1.0),
    feature_dropout: float = 0.10,
    noise_std: float = 0.02,
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    gen = rng if rng is not None else np.random.default_rng()
    total_len = features.shape[0]
    ratio = float(gen.uniform(crop_ratio_range[0], crop_ratio_range[1]))
    win_len = max(1, min(total_len, int(round(total_len * ratio))))
    max_start = total_len - win_len
    start = int(gen.integers(0, max_start + 1)) if max_start > 0 else 0
    end = start + win_len - 1
    indices = np.round(np.linspace(start, end, total_len)).astype(np.int64)
    aug = features[indices].astype(np.float32, copy=True)

    if feature_dropout > 0.0:
        keep_prob = 1.0 - float(feature_dropout)
        mask = (gen.random(aug.shape) < keep_prob).astype(np.float32)
        aug = (aug * mask) / max(keep_prob, 1e-6)
    if noise_std > 0.0:
        aug = aug + gen.normal(0.0, float(noise_std), size=aug.shape).astype(np.float32)
    return aug


class CachedFeatureDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray, augment: bool = False):
        self.features = features
        self.labels = labels
        self.augment = augment

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        seq = np.array(self.features[idx], dtype=np.float32, copy=True)
        if self.augment:
            seq = augment_feature_sequence(seq)
        return torch.from_numpy(seq), torch.tensor(int(self.labels[idx]), dtype=torch.long)


def compute_metrics(y_true: np.ndarray, logits: np.ndarray) -> dict[str, Any]:
    y_t = np.asarray(y_true, dtype=np.int64)
    scores = np.asarray(logits, dtype=np.float64)
    y_pred = np.argmax(scores, axis=1)
    top1 = float(np.mean(y_pred == y_t))
    top5_idx = np.argpartition(-scores, kth=min(4, scores.shape[1] - 1), axis=1)[:, :5]
    top5 = float(np.mean(np.any(top5_idx == y_t[:, None], axis=1)))

    # Macro over present classes
    f1s, precs, recs = [], [], []
    for cid in range(scores.shape[1]):
        tp = int(np.sum((y_t == cid) & (y_pred == cid)))
        fp = int(np.sum((y_t != cid) & (y_pred == cid)))
        fn = int(np.sum((y_t == cid) & (y_pred != cid)))
        sup = tp + fn
        if sup == 0:
            continue
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / sup
        f1 = (2.0 * p * r / (p + r)) if (p + r) > 0 else 0.0
        precs.append(p)
        recs.append(r)
        f1s.append(f1)

    return {
        "samples": int(len(y_t)),
        "top1_accuracy": top1,
        "top5_accuracy": top5,
        "macro_precision": float(np.mean(precs)),
        "macro_recall": float(np.mean(recs)),
        "macro_f1": float(np.mean(f1s)),
        "unique_predicted_classes": int(len(np.unique(y_pred))),
    }


def make_balanced_sampler(labels: np.ndarray, power: float = 0.5) -> WeightedRandomSampler:
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float64)
    cls_w = np.where(counts > 0, 1.0 / np.power(np.maximum(counts, 1.0), power), 0.0)
    sample_w = cls_w[labels]
    return WeightedRandomSampler(weights=sample_w.tolist(), num_samples=len(labels), replacement=True)

## 4. Visual Baseline Model Definitions


In [ ]:
# ==============================================================================
# VISUAL MODEL ARCHITECTURES (IDENTICAL STATE_DICT KEYS TO src/models/visual/)
# ==============================================================================
class TemporalAttentionPool(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.proj = nn.Linear(dim, 1)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.proj(sequence), dim=1)
        return (sequence * weights).sum(dim=1)


class TemporalLSTM(nn.Module):
    def __init__(
        self,
        input_dim: int = 2048,
        hidden_dim: int = 256,
        num_layers: int = 2,
        dropout: float = 0.35,
        bidirectional: bool = False,
        pooling: str = "last",
    ) -> None:
        super().__init__()
        self.input_proj = nn.Sequential(nn.LayerNorm(input_dim), nn.Dropout(dropout))
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.output_dim = hidden_dim * (2 if bidirectional else 1)
        self.pooling = pooling
        self.attn_pool = TemporalAttentionPool(self.output_dim) if pooling == "attention" else None

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(features)
        sequence, (hidden, _) = self.lstm(x)
        if self.pooling == "mean":
            return sequence.mean(dim=1)
        if self.pooling == "attention" and self.attn_pool is not None:
            return self.attn_pool(sequence)
        if self.lstm.bidirectional:
            return torch.cat([hidden[-2], hidden[-1]], dim=-1)
        return hidden[-1]


class TemporalTransformerEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 2048,
        embed_dim: int = 256,
        num_layers: int = 2,
        num_heads: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.35,
        max_seq_len: int = 128,
        pooling: str = "attention",
    ) -> None:
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        pe = torch.zeros(1, max_seq_len, embed_dim)
        pos = torch.arange(0, max_seq_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, embed_dim, 2, dtype=torch.float32) * (-math.log(10000.0) / embed_dim))
        pe[0, :, 0::2] = torch.sin(pos * div)
        pe[0, :, 1::2] = torch.cos(pos * div)
        self.register_buffer("pos_encoding", pe, persistent=False)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.output_dim = embed_dim
        self.pooling = pooling
        self.attn_pool = TemporalAttentionPool(embed_dim) if pooling == "attention" else None

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        t_len = features.size(1)
        x = self.input_proj(features) + self.pos_encoding[:, :t_len, :]
        encoded = self.norm(self.encoder(x))
        if self.pooling == "attention" and self.attn_pool is not None:
            return self.attn_pool(encoded)
        return encoded.mean(dim=1)


class VisualClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        *,
        feature_dim: int = 2048,
        hidden_dim: int = 256,
        num_layers: int = 2,
        dropout: float = 0.35,
        temporal_type: str = "lstm",
        bidirectional: bool = False,
        pooling: str = "last",
        num_heads: int = 4,
    ) -> None:
        super().__init__()
        if temporal_type == "lstm":
            self.temporal = TemporalLSTM(
                input_dim=feature_dim,
                hidden_dim=hidden_dim,
                num_layers=num_layers,
                dropout=dropout,
                bidirectional=bidirectional,
                pooling=pooling,
            )
        else:
            self.temporal = TemporalTransformerEncoder(
                input_dim=feature_dim,
                embed_dim=hidden_dim,
                num_layers=num_layers,
                num_heads=num_heads,
                dim_feedforward=hidden_dim * 2,
                dropout=dropout,
                pooling="mean" if pooling == "last" else pooling,
            )
        self.representation_dim = self.temporal.output_dim
        self.head_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.representation_dim, num_classes)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.head_dropout(self.temporal(inputs)))


class MeanPoolLinearBaseline(nn.Module):
    def __init__(self, feature_dim: int = 2048, num_classes: int = 100, dropout: float = 0.35):
        super().__init__()
        self.norm = nn.LayerNorm(feature_dim)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(feature_dim, num_classes)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.fc(self.drop(self.norm(inputs.mean(dim=1))))

## 5. Train & Benchmark Visual Baselines on Validation


In [ ]:
# ==============================================================================
# STEP 1: RUN NON-NEURAL & LINEAR FLOORS + NEURAL VISUAL BASELINES
# ==============================================================================
def eval_model(model: nn.Module, loader: DataLoader, criterion: nn.Module):
    model.eval()
    total_loss, total_n = 0.0, 0
    logits_all, labels_all = [], []
    with torch.inference_mode():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            out = model(xb)
            loss = criterion(out, yb)
            bs = yb.size(0)
            total_n += bs
            total_loss += float(loss.detach()) * bs
            logits_all.append(out.cpu().numpy())
            labels_all.append(yb.cpu().numpy())
    logits_np = np.concatenate(logits_all, axis=0)
    labels_np = np.concatenate(labels_all, axis=0)
    m = compute_metrics(labels_np, logits_np)
    m["loss"] = total_loss / max(1, total_n)
    return m, logits_np, labels_np


def train_experiment(
    exp_id: str,
    model: nn.Module,
    *,
    model_cfg: dict[str, Any],
    use_augmentation: bool = True,
    balanced_sampling: bool = True,
    label_smoothing: float = 0.05,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    epochs: int = EPOCHS,
    patience: int = PATIENCE,
    save_inference_compatible: bool = True,
) -> dict[str, Any]:
    set_seed(SEED)
    model = model.to(DEVICE)
    run_dir = CKPT_DIR / exp_id
    run_dir.mkdir(parents=True, exist_ok=True)

    sampler = make_balanced_sampler(y_train, power=0.5) if balanced_sampling else None
    tr_ds = CachedFeatureDataset(X_train, y_train, augment=use_augmentation)
    va_ds = CachedFeatureDataset(X_val, y_val, augment=False)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=(sampler is None), sampler=sampler)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_score = (-1.0, float("-inf"))  # (val_top1, -val_loss)
    best_epoch = 0
    best_metrics: dict[str, Any] = {}
    no_improve = 0
    history = []

    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss, tr_hits, tr_n = 0.0, 0, 0
        for xb, yb in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            bs = yb.size(0)
            tr_n += bs
            tr_loss += float(loss.detach()) * bs
            tr_hits += int((logits.argmax(dim=1) == yb).sum().item())

        scheduler.step()
        val_m, _, _ = eval_model(model, va_loader, criterion)
        row = {
            "epoch": epoch,
            "train_loss": tr_loss / tr_n,
            "train_top1": tr_hits / tr_n,
            "val_loss": val_m["loss"],
            "val_top1": val_m["top1_accuracy"],
            "val_top5": val_m["top5_accuracy"],
            "val_macro_f1": val_m["macro_f1"],
            "unique_preds": val_m["unique_predicted_classes"],
            "lr": float(optimizer.param_groups[0]["lr"]),
        }
        history.append(row)

        cur_score = (val_m["top1_accuracy"], -val_m["loss"])
        if cur_score > best_score:
            best_score = cur_score
            best_epoch = epoch
            best_metrics = dict(val_m)
            no_improve = 0
            if save_inference_compatible:
                torch.save(
                    {
                        "epoch": best_epoch,
                        "model_state_dict": model.state_dict(),
                        "val_metrics": best_metrics,
                        "class_to_id": class_to_id,
                        "id_to_class": id_to_class,
                        "config": {
                            "num_frames": 32,
                            "image_size": 224,
                            **model_cfg,
                        },
                    },
                    run_dir / "best_model.pt",
                )
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    elapsed = round(time.time() - t0, 2)
    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)
    (run_dir / "val_metrics.json").write_text(json.dumps(best_metrics, indent=2), encoding="utf-8")
    print(
        f"[{exp_id:24s}] best_ep={best_epoch:2d} | val_top1={best_metrics['top1_accuracy']*100:5.2f}% | "
        f"val_top5={best_metrics['top5_accuracy']*100:5.2f}% | macro_f1={best_metrics['macro_f1']:.4f} | "
        f"uniq_cls={best_metrics['unique_predicted_classes']:3d}/100 | {elapsed}s"
    )
    return {
        "experiment_id": exp_id,
        "best_epoch": best_epoch,
        "elapsed_sec": elapsed,
        "history": history,
        "val_metrics": best_metrics,
        "ckpt_path": str(run_dir / "best_model.pt") if save_inference_compatible else "",
    }


# 0. Sanity baselines (Random & Majority)
tr_counts = np.bincount(y_train, minlength=NUM_CLASSES)
maj_cls = int(np.argmax(tr_counts))
maj_top1 = float(np.mean(y_val == maj_cls))
top5_maj = np.argsort(-tr_counts)[:5]
maj_top5 = float(np.mean(np.isin(y_val, top5_maj)))
print(f"[random_chance           ] val_top1= 1.00% | val_top5= 5.00% | majority_top1={maj_top1*100:.2f}% (class='{id_to_class[maj_cls]}')")

results = [
    {"experiment_id": "random_chance", "model": "UniformRandom", "modality": "none", "fusion_mode": "none", "split": "val",
     "top1_accuracy": 0.01, "top5_accuracy": 0.05, "macro_f1": 0.01, "v2t_recall@1": np.nan, "v2t_recall@5": np.nan,
     "t2v_recall@1": np.nan, "mrr": np.nan, "checkpoint_path": ""},
    {"experiment_id": "majority_class", "model": "MajorityClass", "modality": "none", "fusion_mode": "none", "split": "val",
     "top1_accuracy": maj_top1, "top5_accuracy": maj_top5, "macro_f1": 0.0, "v2t_recall@1": np.nan, "v2t_recall@5": np.nan,
     "t2v_recall@1": np.nan, "mrr": np.nan, "checkpoint_path": ""},
]

# 0c. Mean-pool linear baseline
res_lin = train_experiment(
    "mean_pool_linear",
    MeanPoolLinearBaseline(FEATURE_DIM, NUM_CLASSES, dropout=0.35),
    model_cfg={"temporal": "mean_pool_linear"},
    use_augmentation=True,
    balanced_sampling=True,
    label_smoothing=0.05,
    lr=1e-3,
    save_inference_compatible=False,
)

# 1. Classic ResNet50 + LSTM (last hidden state, raw baseline)
res_lstm_last = train_experiment(
    "visual_lstm_last",
    VisualClassifier(NUM_CLASSES, feature_dim=FEATURE_DIM, hidden_dim=256, num_layers=2, dropout=0.35, temporal_type="lstm", bidirectional=False, pooling="last"),
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 256, "num_layers": 2, "dropout": 0.35, "temporal": "lstm", "bidirectional": False, "pooling": "last"},
    use_augmentation=False,
    balanced_sampling=False,
    label_smoothing=0.0,
    lr=1e-3,
)

# 2. Bidirectional LSTM + Attention Pooling + Augmentation + Smoothing
res_bilstm_attn = train_experiment(
    "visual_bilstm_attn",
    VisualClassifier(NUM_CLASSES, feature_dim=FEATURE_DIM, hidden_dim=256, num_layers=2, dropout=0.35, temporal_type="lstm", bidirectional=True, pooling="attention"),
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 256, "num_layers": 2, "dropout": 0.35, "temporal": "lstm", "bidirectional": True, "pooling": "attention"},
    use_augmentation=True,
    balanced_sampling=True,
    label_smoothing=0.05,
    lr=8e-4,
)

# 3. Temporal Transformer + Attention Pooling + Augmentation + Smoothing
res_trans_attn = train_experiment(
    "visual_transformer_attn",
    VisualClassifier(NUM_CLASSES, feature_dim=FEATURE_DIM, hidden_dim=256, num_layers=2, dropout=0.35, temporal_type="transformer", pooling="attention", num_heads=4),
    model_cfg={"feature_dim": FEATURE_DIM, "hidden_size": 256, "num_layers": 2, "dropout": 0.35, "temporal": "transformer", "bidirectional": False, "pooling": "attention"},
    use_augmentation=True,
    balanced_sampling=True,
    label_smoothing=0.05,
    lr=5e-4,
    weight_decay=5e-4,
)

for res, mdl_name in [
    (res_lin, "MeanPoolLinear"),
    (res_lstm_last, "ResNet50+LSTM(last)"),
    (res_bilstm_attn, "ResNet50+BiLSTM(attn)"),
    (res_trans_attn, "ResNet50+Transformer(attn)"),
]:
    vm = res["val_metrics"]
    results.append({
        "experiment_id": res["experiment_id"],
        "model": mdl_name,
        "modality": "visual_only",
        "fusion_mode": "none",
        "split": "val",
        "top1_accuracy": round(vm["top1_accuracy"], 4),
        "top5_accuracy": round(vm["top5_accuracy"], 4),
        "macro_f1": round(vm["macro_f1"], 4),
        "v2t_recall@1": np.nan,
        "v2t_recall@5": np.nan,
        "t2v_recall@1": np.nan,
        "mrr": np.nan,
        "checkpoint_path": res["ckpt_path"],
    })

## 6. Curves, Results Tracker & Summary Output


In [ ]:
# ==============================================================================
# STEP 2: PLOT LEARNING CURVES, SAVE RESULTS TRACKER & PRINT SUMMARY
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for res, lbl in [
    (res_lin, "MeanPool Linear"),
    (res_lstm_last, "LSTM (last)"),
    (res_bilstm_attn, "BiLSTM (attn)"),
    (res_trans_attn, "Transformer (attn)"),
]:
    hdf = pd.DataFrame(res["history"])
    axes[0].plot(hdf["epoch"], hdf["val_loss"], label=lbl)
    axes[1].plot(hdf["epoch"], hdf["val_top1"] * 100.0, label=lbl)
    axes[2].plot(hdf["epoch"], hdf["val_top5"] * 100.0, label=lbl)

axes[0].set_title("Validation Loss")
axes[1].set_title("Validation Top-1 Accuracy (%)")
axes[2].set_title("Validation Top-5 Accuracy (%)")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plot_path = PLOTS_DIR / "03_visual_baselines_curves.png"
plt.savefig(plot_path, dpi=130)
plt.show()

# Idempotently update results_summary.csv on Drive
res_csv = EXP_DIR / "results_summary.csv"
new_df = pd.DataFrame(results)
if res_csv.is_file():
    old_df = pd.read_csv(res_csv)
    old_df = old_df[~((old_df["experiment_id"].isin(new_df["experiment_id"])) & (old_df["split"] == "val"))]
    merged_df = pd.concat([old_df, new_df], ignore_index=True)
else:
    merged_df = new_df
merged_df.to_csv(res_csv, index=False)

print("=" * 72)
print("NOTEBOOK 03 (VISUAL BASELINES ON VALIDATION) COMPLETED")
print("=" * 72)
display_cols = ["experiment_id", "model", "top1_accuracy", "top5_accuracy", "macro_f1"]
print(new_df[display_cols].to_string(index=False))
print(f"\nSaved comparison plot : {plot_path}")
print(f"Updated results table : {res_csv}")